# Step 6 (re-run) — BERTopic: reproducible, seragam dengan LDA v2

Perbedaan terhadap run BERTopic sebelumnya (123 topik, 39,5% outlier, tabel tahun tidak lengkap):
1. **Korpus sama persis dengan LDA v2** (`df_model_final.csv`, filter jurnal sama: kosong).
2. **Kosakata sama dengan LDA** (5.000 term, unigram-trigram, dihitung di level dokumen), sehingga perbedaan hasil tidak berasal dari kosakata.
3. **Evaluasi sama dengan LDA** (C_v gensim pada korpus n-gram yang sama; diversity = unik/total pada top-10 kata).
4. **Seed dan sensitivitas terdokumentasi**: 3 seed pada baseline, lima konfigurasi klaster pada seed 42.
5. **Dilaporkan dua versi**: hasil alami dan hasil `reduce_topics` ke K yang sama dengan LDA (K=25), karena diversity dan coherence bergantung pada jumlah topik.
6. **Outlier (-1) tidak dibuang**; persentasenya dilaporkan dan tabel tren dihitung lengkap 2015-2025.

Embedding dihitung dari kolom `title` (kalimat utuh, bukan token tanpa stopword), sedangkan representasi topik (c-TF-IDF) dari `tokens_joined`. **Estimasi waktu Colab (GPU T4):** embedding ~3-8 menit, UMAP ~2-4 menit per seed, evaluasi C_v beberapa menit per model. Total sekitar 45-90 menit. Aktifkan GPU: Runtime > Change runtime type.

In [ ]:
!pip install bertopic umap-learn hdbscan sentence-transformers gensim -q

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from gensim import corpora
from gensim.models.coherencemodel import CoherenceModel
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.dimensionality import BaseDimensionalityReduction
warnings.filterwarnings('ignore')

# ============ KONFIGURASI ============
QUICK = False
SEEDS = [42, 123, 2024]
EMBED_MODEL = 'paraphrase-multilingual-MiniLM-L12-v2'   # sama dengan run sebelumnya (kontinuitas)
ALT_EMBED_MODEL = 'intfloat/multilingual-e5-base'        # opsional: robustness (bagian G)
UMAP_PARAMS = dict(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine')
HDBSCAN_FIXED = dict(metric='euclidean', cluster_selection_method='eom', prediction_data=True)
# (min_cluster_size, min_samples); None = default HDBSCAN (min_samples = min_cluster_size)
CLUSTER_CONFIGS = {'mcs30': (30, None), 'mcs50': (50, None), 'mcs100': (100, None), 'mcs200': (200, None),
                   'mcs50_ms10': (50, 10), 'mcs100_ms10': (100, 10)}
BASELINE = 'mcs50'                 # sama dengan run sebelumnya (min_topic_size=50)
K_TARGET = 25                      # jumlah topik LDA final -> pembanding yang adil
TOP_N = 10
VECT_PARAMS = dict(ngram_range=(1, 3), max_df=0.85, min_df=5, max_features=5000)   # IDENTIK dengan LDA baseline
INPUT_CANDIDATES = ['df_model_final.csv', 'df_lda_topics.csv']
EXCLUDE_JOURNAL_PATTERNS = []      # HARUS sama dengan yang dipakai di step5 v2
COH_PROCESSES = 2                  # tidak mengubah nilai C_v, hanya kecepatan
EMB_CACHE = 'embeddings_{}.npy'
# ======================================

if QUICK:
    SEEDS = [42, 123]; CLUSTER_CONFIGS = {'mcs30': (30, None), 'mcs50': (50, None)}; BASELINE = 'mcs30'; K_TARGET = 8

path = next(p for p in INPUT_CANDIDATES if os.path.exists(p))
df = pd.read_csv(path).dropna(subset=['tokens_joined', 'title']).reset_index(drop=True)
for pat in EXCLUDE_JOURNAL_PATTERNS:
    df = df[~df['jurnal'].str.contains(pat, case=False, na=False, regex=False)]
df = df.reset_index(drop=True)
if QUICK: df = df.sample(6000, random_state=42).reset_index(drop=True)
print(f"Input: {path} | dokumen={len(df)} | tahun {df['tahun'].min()}-{df['tahun'].max()}")
assert QUICK or len(df) == 39532 or EXCLUDE_JOURNAL_PATTERNS, "Jumlah dokumen berbeda dari LDA v2 (39.532); cek input/filter."

docs_embed = df['title'].astype(str).tolist()        # untuk embedding
docs_tokens = df['tokens_joined'].astype(str).tolist()   # untuk representasi topik & evaluasi

with open('bertopic_config_v2.json', 'w') as f:
    json.dump(dict(seeds=SEEDS, embed_model=EMBED_MODEL, umap=UMAP_PARAMS, hdbscan_fixed=HDBSCAN_FIXED,
                   cluster_configs={k: list(v) for k, v in CLUSTER_CONFIGS.items()}, baseline=BASELINE,
                   k_target=K_TARGET, top_n=TOP_N, vect_params=VECT_PARAMS, n_docs=len(df),
                   exclude_journal_patterns=EXCLUDE_JOURNAL_PATTERNS, input_file=path), f, indent=2)

## A. Kosakata dan korpus evaluasi (identik dengan LDA baseline)

In [ ]:
vec = CountVectorizer(**VECT_PARAMS)
vec.fit(docs_tokens)
vocab = list(vec.get_feature_names_out())
analyzer = vec.build_analyzer()
texts = [analyzer(d) for d in docs_tokens]
dictionary = corpora.Dictionary(texts)
print(f"Vocabulary (level dokumen) = {len(vocab)} | dictionary gensim = {len(dictionary)}")

def evaluate(topic_words):
    flat = [w for t in topic_words for w in t]
    div = len(set(flat)) / len(flat)
    coh = CoherenceModel(topics=topic_words, texts=texts, dictionary=dictionary,
                         coherence='c_v', processes=COH_PROCESSES).get_coherence()
    return coh, div

## B. Embedding (di-cache; simpan ke Google Drive jika sesi sering terputus)

In [ ]:
def get_embeddings(sentences, model_name):
    cache = EMB_CACHE.format(model_name.replace('/', '_'))
    if os.path.exists(cache):
        E = np.load(cache)
        if len(E) == len(sentences): print('Embedding dimuat dari cache:', cache, E.shape); return E
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer(model_name)
    pref = 'query: ' if 'e5' in model_name else ''
    E = m.encode([pref + s for s in sentences], batch_size=128, show_progress_bar=True, normalize_embeddings=True)
    np.save(cache, E); print('Embedding disimpan:', cache, E.shape); return E

emb = get_embeddings(docs_embed, EMBED_MODEL)

## C. UMAP per seed (dihitung sekali per seed, dipakai semua konfigurasi klaster)

In [ ]:
reduced = {}
for s in SEEDS:
    t0 = time.time()
    reduced[s] = UMAP(random_state=s, **UMAP_PARAMS).fit_transform(emb)
    print(f"UMAP seed={s}: {reduced[s].shape} ({time.time()-t0:.0f}s)")

## D. Fungsi eksperimen

In [ ]:
# language='indonesian' WAJIB: nilai bawaan 'english' menghapus karakter non-alfanumerik (mis. 'fp-growth', 'naïve')
# sehingga tidak cocok dengan kosakata tetap dan c-TF-IDF menjadi tak hingga.
def fit_bt(red, mcs, ms):
    hdb = HDBSCAN(min_cluster_size=mcs, min_samples=ms, **HDBSCAN_FIXED)
    tm = BERTopic(umap_model=BaseDimensionalityReduction(), hdbscan_model=hdb,
                  vectorizer_model=CountVectorizer(vocabulary=vocab, ngram_range=VECT_PARAMS['ngram_range']),
                  language='indonesian', top_n_words=TOP_N, calculate_probabilities=False, verbose=False)
    topics, _ = tm.fit_transform(docs_tokens, embeddings=red)
    return tm, np.array(topics)

def summarize_model(tm, topics, name, seed, variant):
    ids = sorted(t for t in set(topics) if t != -1)
    tw = [[w for w, _ in tm.get_topic(t)][:TOP_N] for t in ids]
    coh, div = evaluate(tw)
    sizes = pd.Series(topics[topics != -1]).value_counts()
    return dict(config=name, seed=seed, variant=variant, n_topics=len(ids), n_docs=len(topics),
                n_outliers=int((topics == -1).sum()), outlier_pct=100 * float((topics == -1).mean()),
                coherence_cv=coh, diversity=div, size_median=float(sizes.median()),
                size_max=int(sizes.max()), size_min=int(sizes.min()),
                largest_topic_share_pct=100 * float(sizes.max() / len(topics)))

## E. Grid: baseline pada 3 seed (alami + reduksi ke K_TARGET), konfigurasi lain pada seed pertama
Hasil disimpan bertahap ke `bertopic_grid_v2.csv`.

In [ ]:
rows, models = [], {}
def save(): pd.DataFrame(rows).to_csv('bertopic_grid_v2.csv', index=False)

for name, (mcs, ms) in CLUSTER_CONFIGS.items():
    seeds_here = SEEDS if name == BASELINE else SEEDS[:1]
    for s in seeds_here:
        t0 = time.time()
        tm, topics = fit_bt(reduced[s], mcs, ms)
        r = summarize_model(tm, topics, name, s, 'natural'); rows.append(r)
        print(f"[{name}] seed={s} natural: topics={r['n_topics']} outlier={r['outlier_pct']:.1f}% coh={r['coherence_cv']:.4f} div={r['diversity']:.3f} ({time.time()-t0:.0f}s)")
        models[(name, s, 'natural')] = (tm, topics); save()
        if name == BASELINE:
            tm_r = BERTopic(umap_model=BaseDimensionalityReduction(), hdbscan_model=HDBSCAN(min_cluster_size=mcs, min_samples=ms, **HDBSCAN_FIXED),
                            vectorizer_model=CountVectorizer(vocabulary=vocab, ngram_range=VECT_PARAMS['ngram_range']),
                            language='indonesian', top_n_words=TOP_N, calculate_probabilities=False, verbose=False)
            tp, _ = tm_r.fit_transform(docs_tokens, embeddings=reduced[s])
            tm_r.reduce_topics(docs_tokens, nr_topics=K_TARGET + 1)   # +1: BERTopic ikut menghitung topik outlier (-1)
            tr = np.array(tm_r.topics_)
            r2 = summarize_model(tm_r, tr, name, s, f'reduced_{K_TARGET}'); rows.append(r2)
            print(f"[{name}] seed={s} reduced : topics={r2['n_topics']} outlier={r2['outlier_pct']:.1f}% coh={r2['coherence_cv']:.4f} div={r2['diversity']:.3f}")
            assert r2['n_topics'] == K_TARGET, f"Reduksi menghasilkan {r2['n_topics']} topik, bukan {K_TARGET}"
            models[(name, s, f'reduced_{K_TARGET}')] = (tm_r, tr); save()

grid = pd.DataFrame(rows)
print(grid.round(4).to_string(index=False))

## F. Ringkasan baseline lintas seed (mean ± SD) dan sensitivitas klaster
Laporkan mean ± SD untuk baseline, bukan nilai satu seed terbaik.

In [ ]:
b = grid[grid.config == BASELINE]
summ = b.groupby('variant').agg(n_topics_mean=('n_topics', 'mean'), outlier_pct_mean=('outlier_pct', 'mean'), outlier_pct_sd=('outlier_pct', 'std'),
                                coh_mean=('coherence_cv', 'mean'), coh_sd=('coherence_cv', 'std'),
                                div_mean=('diversity', 'mean'), div_sd=('diversity', 'std')).round(4)
print(summ.to_string())
summ.to_csv('bertopic_baseline_summary_v2.csv')

print("\nSensitivitas konfigurasi klaster (seed pertama, varian natural):")
sens = grid[(grid.seed == SEEDS[0]) & (grid.variant == 'natural')][['config', 'n_topics', 'outlier_pct', 'coherence_cv', 'diversity', 'largest_topic_share_pct']]
print(sens.round(4).to_string(index=False))
sens.to_csv('bertopic_sensitivity_v2.csv', index=False)

## G. (Opsional) Robustness embedding alternatif — multilingual-e5-base
Jalankan hanya jika waktu cukup. Hasilnya dilaporkan sebagai robustness check, bukan untuk memilih model demi skor tertinggi.

In [ ]:
RUN_ALT_EMBED = False
if RUN_ALT_EMBED:
    emb2 = get_embeddings(docs_embed, ALT_EMBED_MODEL)
    red2 = UMAP(random_state=SEEDS[0], **UMAP_PARAMS).fit_transform(emb2)
    mcs, ms = CLUSTER_CONFIGS[BASELINE]
    tm2, tp2 = fit_bt(red2, mcs, ms)
    r = summarize_model(tm2, tp2, BASELINE + '_' + ALT_EMBED_MODEL.split('/')[-1], SEEDS[0], 'natural')
    print(r); rows.append(r); save()

## H. Model final (seed pertama, konfigurasi baseline)
Ekspor topik alami dan topik hasil reduksi. Perbandingan langsung dengan LDA K=25 memakai varian **reduced**.

In [ ]:
S0 = SEEDS[0]
tm_nat, top_nat = models[(BASELINE, S0, 'natural')]
tm_red, top_red = models[(BASELINE, S0, f'reduced_{K_TARGET}')]
df['bertopic_id'] = top_nat
df['bertopic_id_reduced'] = top_red

def topic_table(tm, topics):
    info = []
    for t in sorted(set(topics)):
        words = [w for w, _ in tm.get_topic(t)][:TOP_N] if t != -1 else []
        info.append(dict(topic=t, size=int((topics == t).sum()), top_words=', '.join(words)))
    return pd.DataFrame(info)

ti_nat, ti_red = topic_table(tm_nat, top_nat), topic_table(tm_red, top_red)
ti_nat.to_csv('bertopic_topic_info_natural_v2.csv', index=False); ti_red.to_csv('bertopic_topic_info_reduced_v2.csv', index=False)
print(f"Natural : {len(ti_nat)-1} topik, outlier = {(top_nat==-1).sum()} ({100*(top_nat==-1).mean():.1f}%)")
print(f"Reduced : {len(ti_red)-1} topik, outlier = {(top_red==-1).sum()} ({100*(top_red==-1).mean():.1f}%)")

# dokumen representatif (reduced): jarak ke pusat klaster pada ruang UMAP
red0 = reduced[S0]; rep_rows = []
for t in sorted(set(top_red) - {-1}):
    idx = np.where(top_red == t)[0]
    cen = red0[idx].mean(axis=0)
    order = idx[np.argsort(np.linalg.norm(red0[idx] - cen, axis=1))][:10]
    for i in order: rep_rows.append(dict(topic=t, tahun=df.loc[i, 'tahun'], jurnal=df.loc[i, 'jurnal'], title_raw=df.loc[i, 'title_raw']))
pd.DataFrame(rep_rows).to_csv('bertopic_representative_docs_v2.csv', index=False)

df.to_csv('df_bertopic_topics_v2.csv', index=False)
print(ti_red.to_string(index=False))
for t in sorted(set(top_red) - {-1}):
    print(f"\n--- BT-{t} (n={int((top_red==t).sum())}) ---")
    for r in [x for x in rep_rows if x['topic'] == t][:4]: print('  -', r['title_raw'])

## I. Penamaan topik (MANUAL) dan tabel topik × tahun (lengkap 2015-2025)
Isi `bt_names` untuk semua topik varian reduced setelah membaca top words dan dokumen representatif. Outlier tetap sebagai baris tersendiri.

In [ ]:
bt_names = {
    # 0: "....",
}
ids_red = sorted(set(top_red) - {-1})
assert len(bt_names) == len(ids_red), f"Lengkapi nama untuk {len(ids_red)} topik (baru terisi {len(bt_names)})."
df['bertopic_name'] = df['bertopic_id_reduced'].map(lambda t: 'Outlier' if t == -1 else bt_names[t])

counts = pd.crosstab(df['tahun'], df['bertopic_name'])
assert counts.values.sum() == len(df) and list(counts.index) == sorted(df['tahun'].unique())
props_all = counts.div(counts.sum(axis=1), axis=0)                                       # penyebut = semua dokumen tahun tsb
cnt_no = counts.drop(columns='Outlier'); props_assigned = cnt_no.div(cnt_no.sum(axis=1), axis=0)   # penyebut = dokumen ber-topik
counts.to_csv('bertopic_topic_by_year_counts_v2.csv')
props_all.to_csv('bertopic_topic_by_year_proportions_all_v2.csv')
props_assigned.to_csv('bertopic_topic_by_year_proportions_assigned_v2.csv')
df.to_csv('df_bertopic_topics_v2.csv', index=False)
print(counts.sum(axis=1)); print("Outlier per tahun (%):"); print((100 * props_all['Outlier']).round(1).to_string())

## J. Figure 4 (distribusi ukuran topik) dan Figure 8 (peta 2D)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, top, ttl in [(axes[0], top_nat, 'Natural'), (axes[1], top_red, f'Reduced to {K_TARGET}')]:
    sz = pd.Series(top[top != -1]).value_counts().sort_values(ascending=False).values
    ax.bar(range(1, len(sz) + 1), sz, color='#4C72B0')
    ax.axhline((top == -1).sum(), color='#C44E52', ls='--', lw=1, label=f'Outliers (n={(top == -1).sum()})')
    ax.set_yscale('log'); ax.set_xlabel('Topic rank by size'); ax.set_ylabel('Number of titles (log)'); ax.set_title(ttl); ax.legend(frameon=False); ax.grid(alpha=.3, axis='y')
fig.suptitle('Figure 4. BERTopic topic size distribution (seed %d, %s)' % (S0, BASELINE), fontsize=11)
fig.tight_layout(); plt.savefig('figure4_bertopic_topic_distribution_v2.png', dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
xy = UMAP(random_state=S0, n_neighbors=15, n_components=2, min_dist=0.1, metric='cosine').fit_transform(emb)
n_show = min(15, len(ids_red)); top_ids = pd.Series(top_red[top_red != -1]).value_counts().index[:n_show]
fig, ax = plt.subplots(figsize=(8, 7))
m = top_red == -1; ax.scatter(xy[m, 0], xy[m, 1], s=1, c='#DDDDDD', label='Outlier', rasterized=True)
cmap = plt.get_cmap('tab20')
for k, t in enumerate(top_ids):
    m = top_red == t; ax.scatter(xy[m, 0], xy[m, 1], s=1.5, color=cmap(k % 20), rasterized=True)
    ax.annotate(f'BT-{t}', xy[m].mean(axis=0), fontsize=8, weight='bold')
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2'); ax.set_title(f'Figure 8. 2D UMAP of title embeddings (top {n_show} topics after reduction; grey = outliers)', fontsize=10)
plt.savefig('figure8_bertopic_umap2d_v2.png', dpi=300, bbox_inches='tight'); plt.show()

## K. (Opsional) Sensitivitas pengurangan outlier
`reduce_outliers` memberi topik pada dokumen outlier sehingga mengubah penugasan. Laporkan hanya sebagai analisis sensitivitas; tren utama tetap memakai hasil tanpa reduksi.

In [ ]:
RUN_REDUCE_OUTLIERS = False
if RUN_REDUCE_OUTLIERS:
    new_topics = tm_red.reduce_outliers(docs_tokens, list(top_red), strategy='embeddings', embeddings=reduced[S0])
    new_topics = np.array(new_topics)
    print('Outlier sebelum:', (top_red == -1).sum(), '| sesudah:', (new_topics == -1).sum())
    df['bertopic_id_reduced_no_outlier'] = new_topics
    pd.crosstab(df['tahun'], df['bertopic_id_reduced_no_outlier']).to_csv('bertopic_topic_by_year_counts_reduced_outliers_v2.csv')